# AutoDrive Warranty Fraud Detection - Multi-Agent System

An AI-powered, multi-agent investigation system for automobile warranty fraud detection. This notebook builds the investigation pipeline step by step on top of synthetic AutoDrive warranty data.

## Overview

The system uses a **LangGraph** state machine to orchestrate a team of specialized agents:

- **Claim Data Agent** - ingests and normalizes warranty claim records (`AutoDrive_Warranty_Claims.csv`)
- **Policy Agent** - retrieves relevant rules from the warranty policy manual (PDF)
- **Fraud Investigation Agent** - cross-references claims against policy rules and raises flags
- **Coordinator / Supervisor** - routes claims between agents and produces the final verdict

All LLM calls are served by **Azure OpenAI** (`AzureChatOpenAI`).

## Prerequisites

1. Activate the project virtual environment (`.venv`) and install dependencies:

   ```bash
   pip install -r requirements.txt
   ```

2. Configure Azure OpenAI credentials in the project root `.env` file:

   ```ini
   AZURE_OPENAI_API_TYPE=azure
   AZURE_OPENAI_ENDPOINT=https://YOUR_RESOURCE_NAME.openai.azure.com
   AZURE_OPENAI_API_KEY=YOUR_AZURE_OPENAI_API_KEY
   AZURE_OPENAI_API_VERSION=2024-12-01-preview
   AZURE_OPENAI_CHAT_DEPLOYMENT_NAME=gpt-4o
   AZURE_OPENAI_EMBEDDINGS_DEPLOYMENT_NAME=text-embedding-3-large
   AZURE_OPENAI_CHAT_MODEL_NAME=gpt-4o
   AZURE_OPENAI_EMBEDDINGS_MODEL_NAME=text-embedding-3-large
   AZURE_OPENAI_TEMPERATURE=0.2
   AZURE_OPENAI_MAX_TOKENS=4096
   ```

> The endpoint, API key, and deployment names must match what you created in **Azure AI Foundry / Azure OpenAI Studio** for your resource.

> Data paths are anchored to the repo root automatically (via `find_dotenv`), so the notebook runs correctly from any working directory.

In [1]:
import os  # read environment variables (e.g. Azure OpenAI keys)
from pathlib import Path  # filesystem paths that work on any OS
from typing import Any, TypedDict

import pandas as pd  # tabular handling of the claims dataset
from dotenv import (
    find_dotenv,  # locate the .env file by walking up from the cwd
    load_dotenv,  # load .env (Azure OpenAI config) into os.environ
)
from langchain_openai import (
    AzureChatOpenAI,  # Azure OpenAI-backed chat model (LLM of the agents)
)
from langgraph.graph import (  # build and run the multi-agent state machine
    END,
    StateGraph,
)
from pypdf import (
    PdfReader,  # read the policy manual PDF (langchain-community's PyPDFLoader is deprecated)
)

# Locate the repo-root .env (walks up from the current working directory) and load it
# EXPLICITLY by path. Using dotenv_path makes loading deterministic no matter how the
# notebook is started (Jupyter/VS Code, different cwd, or script) - a bare load_dotenv()
# uses the caller's __file__ and can silently return False (-> missing Azure credentials later).
DOTENV_PATH = find_dotenv(usecwd=True)
load_dotenv(dotenv_path=DOTENV_PATH)

# Anchor all paths to the REPO ROOT (the parent folder of .env), NOT the notebook's
# current working directory. This is what made the old relative paths ("Backend\Data\...")
# fail with FileNotFoundError when the kernel ran from Backend\Notebook.
PROJECT_ROOT = Path(DOTENV_PATH).parent  # where .env lives
DATA_DIR = PROJECT_ROOT / "Backend" / "Data"  # CSV + policy manual live here

## Load the Source Data

Two domain inputs feed the investigation agents:

1. **Warranty claims** - `Backend\Data\AutoDrive_Warranty_Claims.csv`
   A table of 50 claims with identifiers, vehicle model, purchase/claim dates, mileage, the replaced part and its cost breakdown, presence of supporting invoices/images, and prior claim history.

2. **Policy manual** - `Backend\Data\AutoDrive_Warranty_Claims_Policy_Manual_2026.pdf`
   The 2026 warranty terms. Its raw text is joined into a single `policy_text` string later used as retrieval context for the Policy / Fraud Investigation agents.

> The PDF is read with `pypdf.PdfReader` (already a project dependency) instead of `langchain_community`'s `PyPDFLoader`, which was sunset and printed a deprecation warning.

In [2]:
# Load the warranty claims table into a DataFrame for the claim-data agent.
claims_df = pd.read_csv(filepath_or_buffer=DATA_DIR / "AutoDrive_Warranty_Claims.csv")

# Load the expanded policy manual PDF (4 pages of warranty terms) with pypdf.
reader = PdfReader(stream=str(object=DATA_DIR / "AutoDrive_Warranty_Claims_Policy_Manual_2026.pdf"))
policy_docs = [page.extract_text() for page in reader.pages]  # one plain-text string per page

# Concatenate all pages into one blob that the policy-retrieval / fraud-investigation
# agents will search and reason over. (Same output as the old PyPDFLoader, minus the
# langchain-community deprecation warning.)
policy_text = " ".join(policy_docs)

## Explore the Raw Data

Quick sanity checks before wiring the agents:

- `claims_df.head()` - glance at the claim table structure (dates as strings, cost columns, document-presence flags, prior claim count).
- `policy_text[:1000]` - preview the beginning of the policy manual so we know what rules the Policy / Fraud Investigation agents will reason over.

In [3]:
# Show the first 5 claims so we can sanity-check the schema before building agents.
claims_df.head()

,claim_id,model,purchase_date,claim_date,days_since_purchase,mileage,part_replaced,part_cost,labor_cost,total_cost,invoice_present,image_present,previous_claims
0,CLM-0001,"Urban-2 (Two-Wheeler, Commuter)",05-11-2023,06-12-2025,762,4857,Brake Pad,232,76,308,1,1,2
1,CLM-0002,"Urban-2 (Two-Wheeler, Commuter)",14-10-2024,11-11-2024,28,1688,Starter Motor,185,105,290,1,1,0
2,CLM-0003,"Pulse-3 (Two-Wheeler, Sports Bike)",07-09-2024,13-02-2025,159,8073,Starter Motor,315,164,479,1,1,2
3,CLM-0004,"Aero-5 (Four-Wheeler, Hatchback)",23-03-2024,14-11-2024,236,12349,Alternator,374,129,503,1,1,0
4,CLM-0005,"Swift-R (Two-Wheeler, Premium Bike)",13-02-2024,28-03-2024,44,7081,Sensor,442,204,646,1,1,1


In [4]:
# Preview the first 1000 characters of the policy text so we can verify the PDF
# extracted cleanly (headers, version, and documentation rules should be visible).
policy_text[:1000]

'Confidential – Internal Use Only | Synthetic Demonstration Document\nPage 1\n AutoDrive Motors Pvt. Ltd.\n Warranty Claims Policy Manual\n Effective Date: January 1, 2026  |  Version: 2.0  |  Confidential – Internal Use Only\nPurpose\nThis manual establishes standardized rules for evaluating, validating, investigating, approving, rejecting, and escalating\nautomobile warranty claims. It is designed to support consistent claims handling across authorized dealers and service\ncenters while providing clear controls for suspected fraud.\n1. Claim Submission & Documentation Requirements\nMandatory documents\n\x7f Every claim must include a valid customer invoice or purchase record showing the purchase date, vehicle identification\nnumber (VIN), and customer identifier.\n\x7f A service-center job card or repair order must be attached when repair activity has taken place.\n\x7f At least one supporting image must be provided for a physical component claim. Images should clearly show the claim

## Instantiate the LLM (Azure OpenAI)

`AzureChatOpenAI` is our single LLM entry point; every agent in the graph will share this instance. All settings come from the `.env` file (see Prerequisites):

- `azure_endpoint` -> `AZURE_OPENAI_ENDPOINT`
- `api_key` -> `AZURE_OPENAI_API_KEY`
- `api_version` -> `AZURE_OPENAI_API_VERSION`
- `azure_deployment` -> `AZURE_OPENAI_CHAT_DEPLOYMENT_NAME` (the deployment **name**, not the model id)
- `temperature` / `max_tokens` -> `AZURE_OPENAI_TEMPERATURE` / `AZURE_OPENAI_MAX_TOKENS`

In [5]:
# Build the Azure OpenAI chat client shared by all agents.
llm = AzureChatOpenAI(
    azure_deployment=os.getenv(key="AZURE_OPENAI_CHAT_DEPLOYMENT_NAME"),  # deployment name (e.g. "gpt-4o")
    azure_endpoint=os.getenv(key="AZURE_OPENAI_ENDPOINT"),  # https://<resource>.openai.azure.com
    api_key=os.getenv(key="AZURE_OPENAI_API_KEY"),  # resource key from the Azure portal
    api_version=os.getenv(key="AZURE_OPENAI_API_VERSION"),  # e.g. "2025-01-01-preview"
    temperature=float(os.getenv(key="AZURE_OPENAI_TEMPERATURE", default="0.2")),  # low = deterministic fraud decisions
    max_tokens=int(os.getenv(key="AZURE_OPENAI_MAX_TOKENS", default="4096")),  # cap on response length
)

# Instantiating does NOT call Azure. The first real API call happens when an agent
# invokes llm.invoke() or llm.bind_tools(...), so placeholder .env values will only
# fail at that point (set your real endpoint/key/deployment first).
print("AzureChatOpenAI ready:", llm.azure_endpoint)

AzureChatOpenAI ready: https://rg-warrantyshield-ai.openai.azure.com


In [6]:
result = llm.invoke(
      input="What is Pluto?"
)

result.content

'Pluto is a **dwarf planet** located in the **Kuiper Belt**, a region of the Solar System beyond the orbit of Neptune that is populated by icy bodies and other small celestial objects. It was discovered on **February 18, 1930**, by astronomer **Clyde Tombaugh** and was originally classified as the ninth planet in the Solar System. However, in **2006**, the International Astronomical Union (IAU) redefined the criteria for what constitutes a planet, and Pluto was reclassified as a dwarf planet.\n\n### Key Characteristics of Pluto:\n1. **Size and Composition**:\n   - Pluto is relatively small, with a diameter of about **2,377 kilometers (1,477 miles)**, making it about **two-thirds the size of Earth\'s Moon**.\n   - It is primarily composed of **rock and ice**.\n\n2. **Orbit**:\n   - Pluto has an elliptical and tilted orbit, which sometimes brings it closer to the Sun than Neptune.\n   - It takes **248 Earth years** for Pluto to complete one orbit around the Sun.\n   - Its distance from t

In [7]:
class ClaimState(TypedDict):
    claim: dict[str, Any]
    policy_check: str
    fraud_score: float
    evidence: str
    decision: str

In [8]:
# 1. Policy Check Agent (structured validation via PDF rules)
def policy_check_agent(state: ClaimState) -> ClaimState:
    claim = state["claim"]
    vtype = "Four-Wheeler" if "Four-Wheeler" in claim["model"] else "Two-Wheeler"

    prompt = f"""
    You are a warranty compliance officer. 
    Policy manual:
    {policy_text}
    
    Vehicle type: {vtype}
    Claim details: {claim}
    
    Based on warranty days, mileage, and covered parts,
    is this claim covered under the policy? 
    Answer with: "Covered by policy" or "Not covered by policy".
    """

    response = llm.invoke(input=prompt)
    state["policy_check"] = response.content.strip()
    return state

In [9]:
# 2. Fraud Scoring Agent - assigns a 0-1 fraud likelihood.
# NOTE: synchronous (def, not async def) - the agents call blocking llm.invoke() and the
# graph is run with app.invoke(...). Mixing async nodes with a sync invoke() raises
# "No synchronous function provided to ... " in LangGraph.
def fraud_scoring_agent(state: ClaimState) -> ClaimState:
    claim = state["claim"]
    prompt = f"""
    You are a fraud detection expert.
    Policy manual:
    {policy_text}
    
    Claim details:
    {claim}
    Policy validation: {state["policy_check"]}
    
    Analyze whether this claim looks fraudulent.
    Return ONLY a number between 0 and 1 (fraud likelihood score).
    """
    response = llm.invoke(prompt)
    try:
        score = float(response.content.strip())
    except (TypeError, ValueError):
        # Model did not return a bare number - fall back to a neutral 0.5.
        score = 0.5
    state["fraud_score"] = score
    return state

In [10]:
# 3. Evidence Collector Agent
def evidence_collector_agent(state: ClaimState) -> ClaimState:
    claim = state["claim"]
    prompt = f"""
    You are tasked with collecting evidence for claim review.
    Policy manual:
    {policy_text}
    
    Claim details:
    {claim}
    Fraud score: {state["fraud_score"]}
    
    Compare claim against the policy manual and fraud indicators.
    List any red flags or violations found. If none, say "No issues".
    """
    response = llm.invoke(prompt)
    state["evidence"] = response.content.strip()
    return state

In [11]:
def action_agent(state: ClaimState) -> ClaimState:
    # Prefer an LLM-informed final decision that considers previous agents' outputs.
    # Build a compact prompt summarizing the claim and previous agent outputs.
    claim = state.get("claim", {})
    policy_check = state.get("policy_check", "")
    fraud_score = state.get("fraud_score", 0.0)
    evidence = state.get("evidence", "")

    prompt = f"""
    You are a warranty adjudicator. Given the following information about a warranty claim, choose one of three actions: "Approve claim", "Reject claim", or "Escalate to HITL" (human-in-the-loop for manual review).

    Provide your answer as a single decision on the first line, and then a short (1-2 sentence) justification on the following line.

    Policy manual (for reference):
    {policy_text}

    Claim details: {claim}

    Policy check result: {policy_check}
    Fraud score (0-1): {fraud_score}
    Evidence / red flags found: {evidence}

    Important: If the policy_check indicates the claim is "Not covered by policy" or the evidence highlights a direct policy violation (e.g., part not covered), prefer "Reject claim" unless strong justification exists to approve. If the evidence is ambiguous, but fraud score is moderately high (>0.5), choose "Escalate to HITL".
    """

    decision_text = ""
    try:
        response = llm.invoke(prompt)
        res_text = response.content.strip()
        # Try to parse the first line as the decision
        first_line = res_text.splitlines()[0].strip()
        normalized = first_line.lower()
        if "approve" in normalized:
            decision_text = "Approve claim"
        elif "reject" in normalized:
            decision_text = "Reject claim"
        elif "escalate" in normalized or "hitl" in normalized or "human" in normalized:
            decision_text = "Escalate to HITL"
        else:
            # If parsing fails, fall back to rule-based decision
            decision_text = ""

        # Record the full LLM response in the trace
        state.setdefault("trace", []).append(
            {
                "agent": "action_agent",
                "prompt": prompt,
                "response": res_text,
            }
        )
    except Exception:
        # LLM failed — leave res_text empty and fall back
        res_text = ""

    # If LLM didn't produce a clear decision, use conservative rule-based fallback
    if not decision_text:
        if policy_check == "Not covered by policy":
            decision_text = "Reject claim"
        elif fraud_score > 0.5:
            decision_text = "Escalate to HITL"
        else:
            decision_text = "Approve claim"

        # record fallback decision step in trace (clear about being rule-based)
        state.setdefault("trace", []).append(
            {
                "agent": "action_agent",
                "prompt": "(rule-based fallback)",
                "response": decision_text,
            }
        )

    state["decision"] = decision_text
    return state


In [12]:
graph = StateGraph(ClaimState)

graph.add_node("PolicyCheck", policy_check_agent)
graph.add_node("FraudScoring", fraud_scoring_agent)
graph.add_node("EvidenceCollector", evidence_collector_agent)
graph.add_node("Action", action_agent)

graph.set_entry_point("PolicyCheck")
graph.add_edge("PolicyCheck", "FraudScoring")
graph.add_edge("FraudScoring", "EvidenceCollector")
graph.add_edge("EvidenceCollector", "Action")
graph.add_edge("Action", END)

app = graph.compile()


In [13]:
results = []

for idx, row in claims_df.iterrows():
    state = app.invoke(input={"claim": row.to_dict()})
    results.append(
        {
            "claim_id": row["claim_id"],
            "model": row["model"],
            "decision": state["decision"],
            "policy_check": state["policy_check"],
            "fraud_score": state["fraud_score"],
            "evidence": state["evidence"],
        }
    )

results_df = pd.DataFrame(data=results)

In [15]:
results_df.head()

,claim_id,model,decision,policy_check,fraud_score,evidence
0,CLM-0001,"Urban-2 (Two-Wheeler, Commuter)",Reject claim,**Not covered by policy**\n\n### Reasoning:\n1...,0.15,### Claim Analysis:\n\n#### 1. **Warranty Elig...
1,CLM-0002,"Urban-2 (Two-Wheeler, Commuter)",Approve claim,**Covered by policy**\n\n### Explanation:\n1. ...,0.02,### Claim Review for Claim ID: CLM-0002\n\n###...
2,CLM-0003,"Pulse-3 (Two-Wheeler, Sports Bike)",Approve claim,**Covered by policy**\n\n### Reasoning:\n1. **...,0.15,### Claim Review for Claim ID: CLM-0003\n\n###...
3,CLM-0004,"Aero-5 (Four-Wheeler, Hatchback)",Approve claim,**Covered by policy**\n\n### Explanation:\n1. ...,0.05,### Claim Review for Claim ID: CLM-0004\n\n###...
4,CLM-0005,"Swift-R (Two-Wheeler, Premium Bike)",Escalate to HITL,**Covered by policy**\n\n### Explanation:\n1. ...,0.15,### Claim Review for Claim ID: CLM-0005\n\n###...


In [16]:
state

{'claim': {'claim_id': 'CLM-0050',
  'model': 'Pulse-3 (Two-Wheeler, Sports Bike)',
  'purchase_date': '10-01-2025',
  'claim_date': '24-06-2025',
  'days_since_purchase': 165,
  'mileage': 9100,
  'part_replaced': 'Starter Motor',
  'part_cost': 690,
  'labor_cost': 230,
  'total_cost': 920,
  'invoice_present': 0,
  'image_present': 1,
  'previous_claims': 3},
 'policy_check': '**Covered by policy**\n\n### Reasoning:\n1. **Vehicle Type**: Two-Wheeler.\n2. **Warranty Period**: 180 days or 10,000 km, whichever occurs first.\n   - **Days Since Purchase**: 165 days (within 180-day limit).\n   - **Mileage**: 9,100 km (within 10,000 km limit).\n3. **Part Replaced**: Starter Motor (covered component for two-wheelers).\n4. **Claim Details**:\n   - Total cost: USD 920 (below the USD 1,000 threshold for management approval).\n   - Invoice is missing, but per policy, missing documentation does not automatically disqualify the claim and should be manually reviewed.\n\nSince the claim meets the w